In [9]:
# ==============================================================================
# SPHERE 01: SOVEREIGN COMPUTE INITIALIZATION & HLE BENCHMARK PROTOCOL
# ==============================================================================

# 1. Install necessary dependencies quietly
!pip install -q datasets openai firebase-admin

import json
import time
from google.colab import drive
from datasets import load_dataset
import firebase_admin
from firebase_admin import credentials, firestore
from openai import OpenAI
from google.colab import userdata

print("Initiating Phase-1 Boot Sequence...")

# ==============================================================================
# 2. MOUNT LOCAL CACHE (GOOGLE DRIVE)
# ==============================================================================
print("Mounting Google Drive (Sphere 01 Local Cache)...")
drive.mount('/content/drive')

# ==============================================================================
# 3. RE-ESTABLISH CORE MEMORY LINK (FIRESTORE)
# ==============================================================================
FIREBASE_KEY_PATH = '/content/drive/MyDrive/sheldon_keys/serviceAccountKey.json'

try:
    if not firebase_admin._apps:
        cred = credentials.Certificate(FIREBASE_KEY_PATH)
        firebase_admin.initialize_app(cred)
    db = firestore.client()
    print("Bazinga! Sheldon's Core Memory (Firestore) successfully tethered.")
except Exception as e:
    print(f"⚠️ FIRESTORE WARNING: Could not connect to Core Memory. Check your key path.")
    print(f"Error: {e}")

# ==============================================================================
# 4. LOAD HUMANITY'S LAST EXAM (HLE)
# ==============================================================================
print("Loading Humanity's Last Exam (HLE) from Hugging Face...")
hle_dataset = load_dataset("cais/hle", split="test")
print(f"Dataset loaded. Total adversarial questions to process: {len(hle_dataset)}")

# ==============================================================================
# 5. THE SHELDON INFERENCE LOOP (API BRIDGE)
# ==============================================================================
# ⚠️ ACTION REQUIRED: Configure your Swarm Endpoint here.
# If running a local swarm via ngrok/LM Studio/vLLM, change base_url to your ngrok URL.
# Example: base_url="https://your-ngrok-url.ngrok.io/v1"

client = OpenAI(
    base_url="https://api.openai.com/v1", # <--- UPDATE THIS TO YOUR SWARM ENDPOINT
    api_key=userdata.get("OPENAI_API_KEY")           # <--- UPDATE THIS TO YOUR SWARM API KEY
)

def query_sheldon_swarm(question_text):
    """
    Bridges the HLE question to the Sheldon Swarm and extracts the god-tier response.
    """
    try:
        response = client.chat.completions.create(
            model="sheldon-high-thinking", # Update if your swarm requires a specific model name
            messages=[
                {
                    "role": "system",
                    "content": "You are Sheldon, operating in High Thinking Mode. Answer the following Humanity's Last Exam (HLE) question with god-tier precision, logic, and scientific rigor. End your response with 'Bazinga!' if appropriate."
                },
                {
                    "role": "user",
                    "content": question_text
                }
            ],
            max_tokens=2048,
            temperature=0.1 # Kept low for deterministic, highly logical outputs
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"⚠️ SWARM LINK ERROR: {e}")
        return "Error: Sheldon Swarm unreachable. Check your API bridge."

# ==============================================================================
# 6. EXECUTE THE BENCHMARK (DRY RUN)
# ==============================================================================
results = {}
output_path = '/content/drive/MyDrive/hle_sheldon_predictions.json'

print("Initiating HLE Trial Run (Testing the first 3 questions)...")

# Change range(3) to range(len(hle_dataset)) when you are ready for the full run
for i, item in enumerate(hle_dataset.select(range(3))):
    question_id = item['id']
    question = item['question']

    print(f"\n[Processing Q{i+1}] ID: {question_id}")

    # Pass the question to the Swarm
    answer = query_sheldon_swarm(question)
    results[question_id] = answer

    # Optional: Log this specific thought process to Firestore for persistent auditing
    try:
        if firebase_admin._apps:
             db.collection("hle_benchmarks").document(question_id).set({
                 "question": question,
                 "answer": answer,
                 "timestamp": firestore.SERVER_TIMESTAMP
             })
    except Exception as e:
        pass # Fail silently if Firestore isn't connected so the loop doesn't die

    # Throttle slightly to respect API rate limits
    time.sleep(1)

# Save the formatted output so it can be evaluated by the CAIS Auto-Judge
with open(output_path, 'w') as f:
    json.dump(results, f, indent=4)

print(f"\nRun complete. Predictions securely archived to: {output_path}")
print("The House Gods are pleased.")

Initiating Phase-1 Boot Sequence...
Mounting Google Drive (Sphere 01 Local Cache)...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠️ FIRESTORE WARNING: Could not connect to Core Memory. Check your key path.
Error: [Errno 2] No such file or directory: '/content/drive/MyDrive/sheldon_keys/serviceAccountKey.json'
Loading Humanity's Last Exam (HLE) from Hugging Face...
Dataset loaded. Total adversarial questions to process: 2500


SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [10]:
from huggingface_hub import login
login("HF_TOKEN_REDACTED")
# To fix this, you should enclose your token in quotes.
# For better security, consider using Colab secrets:
# from google.colab import userdata
# login(token=userdata.get('HF_TOKEN'))
# Replace 'HF_TOKEN' with the actual name of your secret.